# Lab 21-02 · Matching Networks

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import match as m

TH = 10 ** (-10 / 20)


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}


def band(elements, zl, f0, lo, hi, n=20001):
    f = np.linspace(lo, hi, n)
    g = np.array([abs(m.gamma_of(m.zin_ladder(elements, zl, ff))) for ff in f])
    ok = f[g <= TH]
    return ok.min(), ok.max(), (ok.max() - ok.min()) / f0

## Part 1 · One L-section

*Predict* the 10 dB bandwidth of Example 21-02.1 from Q = 2.

In [2]:
f0 = 2.4e9
w = 2 * math.pi * f0
x, b, q = m.l_match(10, 50)
net = [("sL", x / w), ("pC", b / w)]
lo, hi, fb = band(net, 10, f0, 1e9, 4e9)
print(f"band {lo / 1e9:.3f}-{hi / 1e9:.3f} GHz, fractional {fb * 100:.1f} %, times Q = {fb * q:.2f}")

band 1.901-2.812 GHz, fractional 38.0 %, times Q = 0.76


## Part 2 · Q on purpose

*Predict* how a π network's bandwidth changes with its Q.

In [3]:
qs, bws = [], []
for qq in (2.5, 3, 4, 5, 7, 10):
    b1, xs, b2, rv = m.pi_match(50, 10, qq)
    net = [("pC", b2 / w), ("sL", xs / w), ("pC", b1 / w)]
    qs.append(qq); bws.append(band(net, 10, f0, 1e9, 4e9)[2] * 100)
plt.figure(); plt.plot(qs, bws, "o-"); plt.xlabel("loaded Q"); plt.ylabel("10 dB bandwidth (%)"); plt.show()
print([f"Q {a}: {bb:.1f} %" for a, bb in zip(qs, bws)])

['Q 2.5: 28.3 %', 'Q 3: 21.2 %', 'Q 4: 13.9 %', 'Q 5: 10.4 %', 'Q 7: 7.0 %', 'Q 10: 4.7 %']


## Part 3 · Sections, and ngspice

*Predict* the bandwidth of one, two and three sections from 5 Ω to 50 Ω at 2 GHz.

In [4]:
f0 = 2e9
w = 2 * math.pi * f0
for n in (1, 2, 3):
    rs, qsec = m.cascade_l(5, 50, n)
    net = []
    for k in range(n):
        xk, bk, _ = m.l_match(rs[k], rs[k + 1])
        net += [("sL", xk / w), ("pC", bk / w)]
    print(f"{n} section(s): Q {qsec:.2f}, bandwidth {band(net, 5, f0, 0.5e9, 4e9)[2] * 100:.1f} %")
print(ngspice("matchnets.cir"))

1 section(s): Q 3.00, bandwidth 23.6 %


2 section(s): Q 1.47, bandwidth 70.5 %
3 section(s): Q 1.07, bandwidth 125.2 %
{'f1lo': 1900888000.0, 'f1hi': 2811873000.0, 'f3lo': 2265776000.0, 'f3hi': 2515677000.0, 'f5lo': 1608887000.0, 'f5hi': 3018491000.0, 'g1lo': 1.900888, 'g1hi': 2.811873, 'g3lo': 2.265776, 'g3hi': 2.515677, 'g5lo': 1.608887, 'g5hi': 3.018491}


## Part 4 · Transformers

*Predict* how much the exact bandwidth differs from the small-reflection estimate.

In [5]:
x = np.linspace(0.01, 1.99, 4000)
plt.figure()
for n in (1, 2, 3):
    zs = m.binomial_transformer(50, 100, n)[0]
    g = [abs(m.gamma_of(m.zin_ladder([("line", z, math.pi / 2, 1.0) for z in reversed(zs)], 100, xx))) for xx in x]
    plt.plot(x, g, label=f"N = {n}, estimate {m.binomial_bandwidth(50, 100, n, 0.05) * 100:.1f} %")
plt.axhline(0.05, color="k", lw=0.6); plt.xlabel("f / f0"); plt.ylabel("|Gamma|"); plt.legend(); plt.show()

## Part 5 · Loss and limits

*Predict* the number of sections that minimizes loss from 2 Ω to 50 Ω with Q_u = 20.

In [6]:
for n in (1, 2, 3, 4):
    rs, qsec = m.cascade_l(2, 50, n)
    print(f"{n} section(s): Q {qsec:.2f}, loss {n * 10 * math.log10(1 + qsec / 20):.3f} dB")
print(f"Bode-Fano bound for 50 ohm || 1 pF at 10 dB return loss: {m.bode_fano_bw(50, 1e-12, TH) / 1e9:.2f} GHz")

1 section(s): Q 4.90, loss 0.952 dB
2 section(s): Q 2.00, loss 0.828 dB
3 section(s): Q 1.39, loss 0.874 dB
4 section(s): Q 1.11, loss 0.940 dB
Bode-Fano bound for 50 ohm || 1 pF at 10 dB return loss: 8.69 GHz
